# Residual mined bigvgan adapter training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook trains the residual-mined BigVGAN adapter.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
# =========================
# Imports
# =========================
import os, sys, json, math, random, subprocess, gc
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.nn.utils import spectral_norm
from IPython.display import Audio, display, Markdown

print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


def torch_load_any(path, map_location="cpu"):
    """Compatibility wrapper for PyTorch versions where torch.load defaults to weights_only=True."""
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)


In [ ]:

# =========================
# Config: long-run training
# BigVGAN-residual mining, regular linear interpolation only
# =========================
DRIVE_ROOT = Path("/content/drive/MyDrive/master")
CHECKPOINTS_RUNS_DIR = DRIVE_ROOT / "checkpoints_runs"
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"

# Controlled 1× inpainting setup.
K = 1
TRAIN_OFFSETS = [0, 1]
EVAL_OFFSET = 0

# BigVGAN / mel frontend defaults. Keep these matched to the BigVGAN model.
SR = 22050
BIGVGAN_MODEL_ID = "nvidia/bigvgan_v2_22khz_80band_fmax8k_256x"
BIGVGAN_USE_CUDA_KERNEL = False
N_FFT = 1024
HOP = 256
WIN = 1024
N_MELS = 80
FMIN = 0
FMAX = 8000
CENTER_MEL = False

# STFT defaults needed by shared helper/model definitions.
CPLX_N_FFT = 1024
CPLX_HOP = 256
CPLX_WIN = 1024
CPLX_CENTER = True
BLEND_N_FFT = 1024
BLEND_HOP = 256
BLEND_WIN = 1024
BLEND_CENTER = True

# Loss/helper defaults needed by shared functions.
HF_START_FRAC = 0.45
HF_END_GAIN = 8.0
HF_RAMP_POWER = 2.0
MASK_DILATE = 2
TDIFF_MASK_DILATE = 3
HF_D_START_FRAC = 0.45
HF_D_MASK_DILATE = 2

# Old mel 2D U-Net architecture defaults. Checkpoint config will override when available.
G_GROUPS = 8
G_BASE = 24
CPLX_BASE = 24
FUSION_BASE = 24
D_BASE = 32
PRIOR_RADIUS = 3
PRIOR_BOUNDARY_GAIN = 1.5
PRIOR_HF_POWER = 1.25
GATE_TEMP = 2.0
FUSION_IN_CH = 7
MASK_TEMP = 1.0
MASK_REAL_MAX = 1.2
MASK_IMAG_SCALE = 0.75
MASK_MODE = "complex"

# Old 2D mel model selection. This model is frozen and used as context.
TARGET_MODEL_NAME = "Old VOCLP plain 2D"
MANUAL_TARGET_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<run_name>/latest.pt")

MODEL_CANDIDATES = [
    dict(name="Old VOCLP plain 2D", prefix=None),
    dict(name="Supervisor HFRT baseline", prefix=None),
    dict(name="Supervisor HFRT strong", prefix=None),
    dict(name="Broader/discrepancy-guided vocoder-focus", prefix=None),
    dict(name="Post-vocoder STFT HF discriminator", prefix=None),
    dict(name="Supervisor HFRAW baseline", prefix=None),
]

# Split names. Training/mining uses train splits only. Validation mining uses val/eval splits only.
SPEECH_TRAIN_SPLIT = "speech_train.txt"
MUSIC_TRAIN_SPLIT = "music_train.txt"
SPEECH_VAL_CANDIDATES = ["speech_val.txt"]
MUSIC_VAL_CANDIDATES = ["music_val.txt"]

# Familiar diagnostic clips. These are NOT used for training, mining, or checkpoint selection.
FINAL_TEST_CASES = [
    dict(label="SPEECH test idx4 · 6s from 0s", split="speech_test.txt", index=4, start_s=0.0, seg_s=6.0, k=1, offset=0),
    dict(label="MUSIC test idx0 · 10s from 20s", split="music_test.txt", index=0, start_s=20.0, seg_s=10.0, k=1, offset=0),
]

# Diagnostic residual mining. Start small. Increase only after the notebook behaves correctly.
SEG_S = 1.25
SPEECH_PROB_MINE = 0.95
DIAG_MINE_TRAIN_CANDIDATES = 2500   # BigVGAN mining is expensive; increase to 5000 only for longer mining runs
DIAG_MINE_VAL_CANDIDATES = 400
TOPK_TRAIN_LISTEN = 6
TOPK_VAL_LISTEN = 4
REBUILD_RESIDUAL_CACHE = True

# What residual score should target.
RESIDUAL_BAND_LOW_HZ = 2500
RESIDUAL_BAND_HIGH_HZ = 11000
RESIDUAL_SCORE_USE_OLD = True
RESIDUAL_SCORE_USE_LINEAR = True

# Optional checks before any long training.
RUN_SINGLE_EXAMPLE_OVERFIT = False      # keep False for long run
SINGLE_OVERFIT_STEPS = 2000
SINGLE_OVERFIT_LR = 3e-4

RUN_SHORT_PILOT_TRAINING = True         # long run: mine residual-hard examples and train adapter
PILOT_STEPS = 60000
PILOT_BATCH_SIZE = 16
PILOT_LR = 2e-4
PILOT_SAVE_EVERY = 1000
PILOT_VAL_EVERY = 1000
N_RESIDUAL_TRAIN_BANK = 768   # train on the strongest mined post-vocoder residual cases
N_RESIDUAL_VAL_BANK = 96

# Adapter architecture. Linear-only input: linear mel, old model mel, old-model delta, mask.
ADAPTER_BASE = 48
ADAPTER_GROUPS = 8
ADAPTER_DROPOUT = 0.03
ADAPTER_IN_CH = 4

# Loss weights for adapter/pilot. This is supervised on true missing mel frames.
LOSS_W_L1 = 1.0
LOSS_W_L2 = 0.20
LOSS_W_HF = 4.0
LOSS_W_VHF = 3.0
LOSS_W_TDIFF = 2.2
LOSS_W_DDIFF = 1.6
LOSS_W_DELTA = 1.2
LOSS_W_PEAK = 1.5
LOSS_W_RESID_REG = 0.01

OVERFIT_HF_START_FRAC = 0.45
OVERFIT_VHF_START_FRAC = 0.70
OVERFIT_HF_END_GAIN = 10.0
OVERFIT_HF_POWER = 2.0
LOCAL_FRAME_GAIN = 10.0
LOCAL_FRAME_POWER = 1.5
PEAK_TOP_FRAC = 0.06

# Output paths.
RUN_NAME = "residual_mined_bigvgan_adapter"
RUN_DIR = DRIVE_ROOT / "adapter_runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESIDUAL_CACHE_PATH = RUN_DIR / "bigvgan_residual_mining_cache.pt"
LOG_PATH = RUN_DIR / "pilot_train_log.csv"
SAVE_AUDIO_FILES = True
OUT_DIR = RUN_DIR / "audio_debug"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility.
SEED = 20260517
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for p in [DRIVE_ROOT, CHECKPOINTS_RUNS_DIR, SPLIT_DIR, RUN_DIR]:
    print(p, "exists:", p.exists())
print("RUN_DIR:", RUN_DIR)
print("RESIDUAL_CACHE_PATH:", RESIDUAL_CACHE_PATH)


In [ ]:

# =========================
# BigVGAN repo / imports
# =========================
BIGVGAN_REPO = Path("/content/BigVGAN")
if not BIGVGAN_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/NVIDIA/BigVGAN.git", "/content/BigVGAN"], check=True)

if str(BIGVGAN_REPO) not in sys.path:
    sys.path.insert(0, str(BIGVGAN_REPO))

import bigvgan
from meldataset import mel_spectrogram as bigvgan_mel_spectrogram

bigvgan_G = None

def ensure_bigvgan_loaded():
    global bigvgan_G
    if bigvgan_G is not None:
        return bigvgan_G

    print("Loading BigVGAN:", BIGVGAN_MODEL_ID)
    bigvgan_G = bigvgan.BigVGAN._from_pretrained(
        model_id=BIGVGAN_MODEL_ID,
        revision="main",
        cache_dir=None,
        force_download=False,
        proxies=None,
        resume_download=False,
        local_files_only=False,
        token=None,
        map_location=str(device),
        strict=False,
        use_cuda_kernel=BIGVGAN_USE_CUDA_KERNEL,
    ).to(device).eval()

    for p in bigvgan_G.parameters():
        p.requires_grad_(False)

    return bigvgan_G

def wav_to_bigvgan_mel(wav_bt: torch.Tensor):
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    return bigvgan_mel_spectrogram(
        wav_bt,
        n_fft=N_FFT,
        num_mels=N_MELS,
        sampling_rate=SR,
        hop_size=HOP,
        win_size=WIN,
        fmin=FMIN,
        fmax=FMAX,
        center=CENTER_MEL,
    )

def mel_to_wave_bigvgan(mel_bt: torch.Tensor):
    Gv = ensure_bigvgan_loaded()
    y = Gv(mel_bt)
    return y.squeeze(1) if (y.ndim == 3 and y.shape[1] == 1) else y

print("BigVGAN repo ready at:", BIGVGAN_REPO)



# =========================
# Dataset / loaders
# =========================
def read_list(p: Path):
    assert p.exists(), f"Missing list: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav



In [ ]:

# =========================
# Helpers: masks, weights, mel/STFT corruption, losses
# =========================
def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

HF_FREQ_WEIGHTS = build_freq_weights(N_MELS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)
CPLX_FREQ_WEIGHTS = build_freq_weights(CPLX_N_FFT // 2 + 1, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)

def dilate_mask_time(mask: torch.Tensor, radius: int):
    if radius <= 0:
        return mask
    return F.max_pool2d(mask, kernel_size=(1, 2 * radius + 1), stride=1, padding=(0, radius))

def expand_mask(mask: torch.Tensor, n_bins: int, radius: int = 0):
    m = dilate_mask_time(mask, radius)
    return m[:, 0].expand(-1, n_bins, -1)

def masked_mean(x: torch.Tensor, mask_ft: torch.Tensor, freq_weights=None, eps: float = 1e-8):
    z = x.abs()
    if freq_weights is not None:
        z = z * freq_weights
        denom = (mask_ft * freq_weights).sum()
    else:
        denom = mask_ft.sum()
    return (z * mask_ft).sum() / (denom + eps)

def masked_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    return masked_mean(pred - tgt, m, freq_weights=freq_weights)

def masked_tdiff_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    dp = pred[..., 1:] - pred[..., :-1]
    dt = tgt[..., 1:] - tgt[..., :-1]
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)[..., 1:]
    return masked_mean(dp - dt, m, freq_weights=freq_weights)

def masked_complex_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    diff = torch.complex(pred.real - tgt.real, pred.imag - tgt.imag)
    return masked_mean(diff, m, freq_weights=freq_weights)

def masked_logmag_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    pm = torch.log1p(pred.abs())
    tm = torch.log1p(tgt.abs())
    return masked_l1(pm, tm, mask, freq_weights=freq_weights, mask_dilate=mask_dilate)

def linear_time_inpaint_mel(mel: torch.Tensor, k: int, offset: int):
    B, F, T = mel.shape
    step = k + 1
    mel_interp = mel.clone()
    mask = torch.zeros((B, 1, 1, T), device=mel.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            mel_interp[:, :, t] = (1.0 - alpha) * mel[:, :, left] + alpha * mel[:, :, right]
            mask[:, :, :, t] = 1.0
    return mel_interp, mask

def stft_complex(wav_bt: torch.Tensor):
    window = torch.hann_window(CPLX_WIN, device=wav_bt.device)
    return torch.stft(
        wav_bt,
        n_fft=CPLX_N_FFT,
        hop_length=CPLX_HOP,
        win_length=CPLX_WIN,
        window=window,
        center=CPLX_CENTER,
        return_complex=True,
    )

def istft_complex(X_bft: torch.Tensor, length: int):
    window = torch.hann_window(CPLX_WIN, device=X_bft.device)
    return torch.istft(
        X_bft,
        n_fft=CPLX_N_FFT,
        hop_length=CPLX_HOP,
        win_length=CPLX_WIN,
        window=window,
        center=CPLX_CENTER,
        length=length,
    )

def linear_time_inpaint_complex(X: torch.Tensor, k: int, offset: int):
    B, Freq, T = X.shape
    step = k + 1
    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0
    return torch.complex(Xr, Xi), mask

def make_shared_pair(wav_bt: torch.Tensor, k_choices=(1, 2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0

    mel_real = wav_to_bigvgan_mel(wav_bt)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=k, offset=offset)

    X_real = stft_complex(wav_bt)
    X_interp, mask_cplx = linear_time_inpaint_complex(X_real, k=k, offset=offset)

    return {
        "wav": wav_bt,
        "mel_real": mel_real,
        "mel_interp": mel_interp,
        "mask_mel": mask_mel,
        "X_real": X_real,
        "X_interp": X_interp,
        "mask_cplx": mask_cplx,
        "k": k,
        "offset": offset,
    }

def make_hf_prior(mask_mel: torch.Tensor, n_mels: int, radius: int = PRIOR_RADIUS, boundary_gain=None, hf_power=None):
    base = expand_mask(mask_mel, n_mels, radius=0)
    wide = expand_mask(mask_mel, n_mels, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    if boundary_gain is None:
        boundary_gain = PRIOR_BOUNDARY_GAIN
    if hf_power is None:
        hf_power = PRIOR_HF_POWER

    hf = HF_FREQ_WEIGHTS.expand(wide.shape[0], -1, wide.shape[-1])
    hf_norm = ((hf - 1.0) / max(1e-6, (HF_END_GAIN - 1.0))).clamp(0.0, 1.0)
    hf_shape = hf_norm ** hf_power

    region = (base + boundary_gain * boundary).clamp(0.0, 1.0 + boundary_gain)
    return (region * hf_shape).clamp(0.0, 1.0 + PRIOR_BOUNDARY_GAIN)

def make_gate_target(mel_main_hat: torch.Tensor, mel_complex_hat: torch.Tensor, mel_real: torch.Tensor, prior: torch.Tensor):
    err_main = (mel_main_hat - mel_real).abs()
    err_cplx = (mel_complex_hat - mel_real).abs()
    gain = (err_main - err_cplx).clamp_min(0.0) * prior
    den = gain.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6)
    target = (gain / den).clamp(0.0, 1.0)
    return target ** 0.75

def make_adv_focus_view(mel_bt: torch.Tensor, mask_mel: torch.Tensor):
    start_bin = int(round(HF_D_START_FRAC * mel_bt.shape[1]))
    mel_hf = mel_bt[:, start_bin:, :]
    mask_hf = expand_mask(mask_mel, mel_bt.shape[1], radius=HF_D_MASK_DILATE)[:, start_bin:, :]
    return mel_hf * mask_hf, mask_hf

def get_roundtrip_views(mel_hat: torch.Tensor, mel_real: torch.Tensor, mask_mel: torch.Tensor, with_grad_fake: bool = True):
    sb = min(ROUNDTRIP_SUBBATCH, mel_hat.shape[0])
    mel_hat_sb = mel_hat[:sb]
    mel_real_sb = mel_real[:sb]
    mask_sb = mask_mel[:sb]

    if ADV_VIEW_MODE == "raw_mel":
        mel_view_hat = mel_hat_sb
        mel_view_real = mel_real_sb
    else:
        if with_grad_fake:
            wav_hat = mel_to_wave_bigvgan(mel_hat_sb)
        else:
            with torch.no_grad():
                wav_hat = mel_to_wave_bigvgan(mel_hat_sb)

        with torch.no_grad():
            wav_real = mel_to_wave_bigvgan(mel_real_sb)

        mel_view_hat = wav_to_bigvgan_mel(wav_hat)
        mel_view_real = wav_to_bigvgan_mel(wav_real)

    T = min(mel_view_hat.shape[-1], mel_view_real.shape[-1], mask_sb.shape[-1])
    mel_view_hat = mel_view_hat[..., :T]
    mel_view_real = mel_view_real[..., :T]
    mask_sb = mask_sb[..., :T]

    return mel_view_hat, mel_view_real, mask_sb


def _match_frames(mel_bt: torch.Tensor, target_T: int):
    cur_T = int(mel_bt.shape[-1])
    if cur_T == target_T:
        return mel_bt
    if cur_T > target_T:
        return mel_bt[..., :target_T]
    return F.pad(mel_bt, (0, target_T - cur_T))

def roundtrip_mel_from_mel(mel_bt: torch.Tensor):
    wav_bt = mel_to_wave_bigvgan(mel_bt)
    mel_rt = wav_to_bigvgan_mel(wav_bt)
    return _match_frames(mel_rt, mel_bt.shape[-1])

def stft_complex_cfg(wav_bt: torch.Tensor, n_fft: int, hop: int, win: int, center: bool):
    window = torch.hann_window(win, device=wav_bt.device)
    return torch.stft(
        wav_bt, n_fft=n_fft, hop_length=hop, win_length=win,
        window=window, center=center, return_complex=True,
    )

def istft_complex_cfg(X_bft: torch.Tensor, n_fft: int, hop: int, win: int, center: bool, length: int):
    """Robust ISTFT wrapper.

    PyTorch can throw a window-overlap-add error for Hann windows with center=False,
    especially when we create new stretched STFT lengths. The training notebooks used
    different center settings across experiments, so for evaluation we first try the
    requested setting and then fall back to center=True if this specific COLA/WOLA
    error appears.
    """
    window = torch.hann_window(win, device=X_bft.device)

    def _call(center_flag: bool):
        return torch.istft(
            X_bft, n_fft=n_fft, hop_length=hop, win_length=win,
            window=window, center=center_flag, length=length,
        )

    try:
        return _call(center)
    except RuntimeError as e:
        msg = str(e)
        if ("window overlap add" in msg or "overlap add" in msg) and center is False:
            print("[ISTFT fallback] center=False caused a window-overlap-add error; retrying center=True for evaluation.")
            return _call(True)
        raise

def linear_time_inpaint_complex_cfg(X: torch.Tensor, k: int, offset: int):
    B, Freq, T = X.shape
    step = k + 1
    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)
    if k == 0:
        return torch.complex(Xr, Xi), mask
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0
    return torch.complex(Xr, Xi), mask

def trim_to_common_length(*xs):
    lens = [x.shape[-1] for x in xs if x is not None]
    T = min(lens)
    out = []
    for x in xs:
        out.append(None if x is None else x[..., :T])
    return out

def plain_mask_to_ft(mask: torch.Tensor, n_bins: int):
    return mask.expand(-1, n_bins, -1)

def exact_summary_row(model_name: str, family: str, out_kind: str, mel_hat: torch.Tensor, mel_real: torch.Tensor, mask_mel: torch.Tensor, extra=None):
    mel_base, mel_hat, mel_real, mask_mel = trim_to_common_length(CLIP_CTX["mel_interp"], mel_hat, mel_real, mask_mel)
    base_recon = masked_l1(mel_base, mel_real, mask_mel).item()
    ref_recon  = masked_l1(mel_hat, mel_real, mask_mel).item()
    base_hf = masked_l1(mel_base, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    ref_hf  = masked_l1(mel_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    base_tdiff = masked_tdiff_l1(mel_base, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()
    ref_tdiff  = masked_tdiff_l1(mel_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()
    mel_rt_real = roundtrip_mel_from_mel(mel_real)
    mel_rt_hat  = roundtrip_mel_from_mel(mel_hat)
    mel_rt_hat, mel_rt_real, mask_rt = trim_to_common_length(mel_rt_hat, mel_rt_real, mask_mel)
    rt_hf = masked_l1(mel_rt_hat, mel_rt_real, mask_rt, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    rt_tdiff = masked_tdiff_l1(mel_rt_hat, mel_rt_real, mask_rt, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()
    row = dict(
        model=model_name,
        family=family,
        output_kind=out_kind,
        base_recon=base_recon, ref_recon=ref_recon,
        base_hf=base_hf, ref_hf=ref_hf,
        base_tdiff=base_tdiff, ref_tdiff=ref_tdiff,
        roundtrip_hf=rt_hf, roundtrip_tdiff=rt_tdiff,
    )
    if extra:
        row.update(extra)
    return row


# =========================
# Extra helpers for complex-mask mel/STFT fusion checkpoints
# =========================
def match_frames_tensor(x: torch.Tensor, T: int):
    if x.shape[-1] > T:
        return x[..., :T]
    if x.shape[-1] < T:
        return F.pad(x, (0, T - x.shape[-1]))
    return x

def match_mask_frames(mask: torch.Tensor, T: int):
    if mask.shape[-1] > T:
        return mask[..., :T]
    if mask.shape[-1] < T:
        return F.pad(mask, (0, T - mask.shape[-1]))
    return mask

def trim_complex_to_common(*xs):
    T = min(x.shape[-1] for x in xs)
    return [x[..., :T] for x in xs]

def match_wav_length(wav_bt: torch.Tensor, target_len: int):
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    if wav_bt.shape[-1] > target_len:
        wav_bt = wav_bt[..., :target_len]
    elif wav_bt.shape[-1] < target_len:
        wav_bt = F.pad(wav_bt, (0, target_len - wav_bt.shape[-1]))
    return wav_bt

def safe_logmag_complex(X: torch.Tensor, eps: float = 1e-5):
    return torch.log(X.abs().clamp_min(eps))

def make_stft_prior_eval(mask_stft: torch.Tensor, n_bins: int, freq_weights: torch.Tensor,
                         radius: int, boundary_gain: float, hf_power: float,
                         stft_hf_end_gain: float):
    # mask_stft shape: (B, 1, 1, T)
    base = expand_mask(mask_stft, n_bins, radius=0)
    wide = expand_mask(mask_stft, n_bins, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    fw = freq_weights.to(mask_stft.device).expand(base.shape[0], -1, base.shape[-1])
    hf_norm = ((fw - 1.0) / max(1e-6, stft_hf_end_gain - 1.0)).clamp(0.0, 1.0)
    hf_shape = hf_norm ** hf_power

    region = (base + boundary_gain * boundary).clamp(0.0, 1.0)
    return (region * hf_shape).clamp(0.0, 1.0)

def build_complexmask_fusion_features(X_mel: torch.Tensor, X_stft: torch.Tensor,
                                      prior: torch.Tensor, mask_stft: torch.Tensor,
                                      fusion_in_ch: int = 7):
    X_mel, X_stft = trim_complex_to_common(X_mel, X_stft)
    T = X_mel.shape[-1]
    prior = match_frames_tensor(prior, T)
    mask_ft = expand_mask(match_mask_frames(mask_stft, T), X_mel.shape[1], radius=0)

    lm_mel = safe_logmag_complex(X_mel)
    lm_stft = safe_logmag_complex(X_stft)
    lm_diff = lm_stft - lm_mel

    phase_diff = torch.angle(X_stft) - torch.angle(X_mel)
    cos_d = torch.cos(phase_diff)
    sin_d = torch.sin(phase_diff)

    feat = torch.stack([lm_mel, lm_stft, lm_diff, cos_d, sin_d, prior, mask_ft], dim=1)
    feat[:, 0:3] = feat[:, 0:3].clamp(-12.0, 4.0)
    assert feat.ndim == 4 and feat.shape[1] == fusion_in_ch, f"bad fusion feature shape: {tuple(feat.shape)}"
    return feat, prior, mask_ft

def blend_with_complex_mask_eval(X_mel: torch.Tensor, X_stft: torch.Tensor, C: torch.Tensor, prior: torch.Tensor):
    X_mel, X_stft = trim_complex_to_common(X_mel, X_stft)
    T = X_mel.shape[-1]
    C = match_frames_tensor(C, T)
    prior = match_frames_tensor(prior, T)
    return X_mel + prior * C * (X_stft - X_mel)

def extract_state_dict(ckpt, preferred_keys=("G", "model", "model_state_dict", "state_dict", "generator")):
    if isinstance(ckpt, dict):
        for k in preferred_keys:
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        if all(torch.is_tensor(v) for v in ckpt.values()):
            return ckpt
    raise KeyError(f"Could not find model state dict. Available keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

def cfg_get(cfg, keys, default=None):
    for key in keys:
        if isinstance(key, tuple):
            cur = cfg
            ok = True
            for part in key:
                if isinstance(cur, dict) and part in cur:
                    cur = cur[part]
                else:
                    ok = False
                    break
            if ok:
                return cur
        else:
            if isinstance(cfg, dict) and key in cfg:
                return cfg[key]
    return default

def infer_base_from_state_dict(sd, default=24):
    for key in ["enc1.net.0.net.0.weight", "enc1.block.0.weight"]:
        if key in sd:
            return int(sd[key].shape[0])
    for k, v in sd.items():
        if k.endswith("weight") and getattr(v, "ndim", 0) == 4:
            return int(v.shape[0])
    return default


In [ ]:

# =========================
# Models: mel branch, complex branch, fusion gate, HF discriminator
# =========================
def _valid_groups(ch: int, requested: int):
    for g in [requested, 8, 4, 2, 1]:
        if ch % g == 0:
            return g
    return 1

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8, act="lrelu"):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        act_layer = nn.LeakyReLU(0.2, inplace=True) if act == "lrelu" else nn.SiLU(inplace=True)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            act_layer,
        )
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8, act="lrelu"):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, groups=groups, act=act),
            ConvGNAct(out_ch, out_ch, groups=groups, act=act),
        )
    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2,2), groups=8, act="lrelu"):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        act_layer = nn.LeakyReLU(0.2, inplace=True) if act == "lrelu" else nn.SiLU(inplace=True)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            act_layer,
            ConvGNAct(out_ch, out_ch, groups=groups, act=act),
        )
    def forward(self, x):
        return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8, act="lrelu"):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups, act=act)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)

class MelBranchUNet2D(nn.Module):
    def __init__(self, n_mels, base=24, use_mask=True, groups=8):
        super().__init__()
        self.use_mask = use_mask
        c_in = 1 + (1 if use_mask else 0)
        self.enc1 = DoubleConv(c_in, base, groups=groups)
        self.enc2 = DownBlock(base, base*2, stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2, base*4, stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4, base*4, stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4, base*4, groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base, groups=groups)
        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)
        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False) + s1
        delta = self.out_conv(u0).squeeze(1)
        return delta

class ComplexConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ComplexBranchUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=24, groups=8):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))
        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask.expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)

class FusionUNet2D(nn.Module):
    def __init__(self, in_ch=5, base=24, groups=8):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base, groups=groups)
        self.enc2 = DownBlock(base, base*2, stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2, base*4, stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4, base*4, stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4, base*4, groups=groups)
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base, groups=groups)
        self.gate_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.delta_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
        nn.init.zeros_(self.delta_head.weight); nn.init.zeros_(self.delta_head.bias)

    def forward(self, mel_interp, mel_main, mel_complex, prior, mask):
        x = torch.stack([
            mel_interp,
            mel_main,
            mel_complex,
            prior,
            mask.expand(-1, mel_interp.shape[1], -1),
        ], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False) + s1
        gate_logits = self.gate_head(u0).squeeze(1)
        gate = torch.sigmoid(GATE_TEMP * gate_logits)
        delta = self.delta_head(u0).squeeze(1)
        return gate, delta


# Backward-compatible aliases used by the different training notebooks/checkpoints.
PlainMelRefinerUNet2D = MelBranchUNet2D
ComplexSTFTUNet2D = ComplexBranchUNet2D

class HFDisc(nn.Module):
    def __init__(self, in_ch=2, base=32):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(spectral_norm(nn.Conv2d(in_ch, base, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base, base*2, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base*2, base*4, 5, stride=2, padding=2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(spectral_norm(nn.Conv2d(base*4, base*4, 3, stride=1, padding=1)), nn.LeakyReLU(0.2, inplace=True)),
        ])
        self.out = spectral_norm(nn.Conv2d(base*4, 1, 3, stride=1, padding=1))

    def forward(self, x):
        feats = []
        h = x
        for blk in self.blocks:
            h = blk(h)
            feats.append(h)
        logit = self.out(h)
        return logit, feats




class TemporalBiGRUBottleneck(nn.Module):
    """BiGRU over the time axis at the U-Net bottleneck."""
    def __init__(self, channels: int, freq_bins: int, hidden: int = 256, layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.channels = channels
        self.freq_bins = freq_bins
        self.input_size = channels * freq_bins
        self.gru = nn.GRU(
            input_size=self.input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.proj = nn.Linear(2 * hidden, self.input_size)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(self.input_size)

    def forward(self, x):
        B, C, Fq, T = x.shape
        if Fq != self.freq_bins:
            raise RuntimeError(f"Temporal bottleneck expected freq_bins={self.freq_bins}, got {Fq}. Check N_FFT/pooling.")
        seq = x.permute(0, 3, 1, 2).contiguous().view(B, T, C * Fq)
        y, _ = self.gru(seq)
        y = self.proj(y)
        y = self.norm(seq + self.drop(y))
        y = y.view(B, T, C, Fq).permute(0, 2, 3, 1).contiguous()
        return y

class ComplexSTFTCRNUNet2D(nn.Module):
    """DCCRN-lite inspired complex STFT model."""
    def __init__(self, in_ch=3, out_ch=2, base=16, groups=8, bottleneck_freq_bins=64,
                 crn_hidden=256, crn_layers=1, crn_dropout=0.1):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.temporal = TemporalBiGRUBottleneck(
            channels=base*8,
            freq_bins=bottleneck_freq_bins,
            hidden=crn_hidden,
            layers=crn_layers,
            dropout=crn_dropout,
        )
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))
        xb = self.temporal(xb)
        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

class ComplexMaskFusionUNet2D(nn.Module):
    """
    Complex-mask fusion model used by notebooks 44/45/46.

    Inputs are seven STFT-grid feature maps:
      logmag(mel route), logmag(STFT route), logmag difference,
      cos phase diff, sin phase diff, prior, mask.

    Output is C = alpha + i beta.
    """
    def __init__(self, in_ch=7, base=24, groups=8,
                 mask_temp=1.0, mask_real_max=1.2, mask_imag_scale=0.75,
                 mask_mode="complex"):
        super().__init__()
        self.mask_temp = float(mask_temp)
        self.mask_real_max = float(mask_real_max)
        self.mask_imag_scale = float(mask_imag_scale)
        self.mask_mode = str(mask_mode)

        self.enc1 = DoubleConv(in_ch,       base,     groups=groups)
        self.enc2 = DownBlock(base,         base*2,   stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2,       base*4,   stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4,       base*4,   stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4,      base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.real_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.imag_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)

    def forward(self, feat_bctf):
        s1 = self.enc1(feat_bctf)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1

        real_raw = self.real_head(u0).squeeze(1)
        imag_raw = self.imag_head(u0).squeeze(1)

        alpha = self.mask_real_max * torch.sigmoid(self.mask_temp * real_raw)
        if self.mask_mode == "real_only":
            beta = torch.zeros_like(alpha)
        else:
            beta = self.mask_imag_scale * torch.tanh(imag_raw)

        C = torch.complex(alpha, beta)
        return C, alpha, beta


In [ ]:

# =========================
# Checkpoint selection and shared utilities
# =========================
def find_latest_ckpt(prefix: str):
    cands = sorted(CHECKPOINTS_RUNS_DIR.glob(f"{prefix}_*/latest.pt"))
    cands = [p for p in cands if p.exists()]
    return cands[-1] if cands else None

resolved_plain2d = []
for cand in MODEL_CANDIDATES:
    ckpt = Path(MANUAL_TARGET_CKPT) if (MANUAL_TARGET_CKPT is not None and cand["name"] == TARGET_MODEL_NAME) else find_latest_ckpt(cand["prefix"])
    resolved_plain2d.append({**cand, "ckpt_path": ckpt, "found": ckpt is not None and Path(ckpt).exists()})

df_ckpt = pd.DataFrame([{
    "name": r["name"],
    "prefix": r["prefix"],
    "found": r["found"],
    "ckpt_path": None if r["ckpt_path"] is None else str(r["ckpt_path"]),
} for r in resolved_plain2d])
display(df_ckpt)

if MANUAL_TARGET_CKPT is not None:
    selected_spec = dict(name="MANUAL old mel 2D U-Net", prefix=None, ckpt_path=Path(MANUAL_TARGET_CKPT), found=Path(MANUAL_TARGET_CKPT).exists())
else:
    selected_spec = next((r for r in resolved_plain2d if r["name"] == TARGET_MODEL_NAME and r["found"]), None)
    if selected_spec is None:
        selected_spec = next((r for r in resolved_plain2d if r["found"]), None)
        print("TARGET_MODEL_NAME was not found. Falling back to:", None if selected_spec is None else selected_spec["name"])

assert selected_spec is not None and selected_spec["found"], "No plain2d checkpoint found. Set MANUAL_TARGET_CKPT."
print("Selected:", selected_spec["name"])
print("Checkpoint:", selected_spec["ckpt_path"])


def first_existing_split(cands):
    for name in cands:
        if (SPLIT_DIR / name).exists():
            return name
    return None

SPEECH_VAL_SPLIT = first_existing_split(SPEECH_VAL_CANDIDATES)
MUSIC_VAL_SPLIT = first_existing_split(MUSIC_VAL_CANDIDATES)
print("Validation splits:", SPEECH_VAL_SPLIT, MUSIC_VAL_SPLIT)
assert (SPLIT_DIR / SPEECH_TRAIN_SPLIT).exists(), f"Missing {SPEECH_TRAIN_SPLIT}"
assert (SPLIT_DIR / MUSIC_TRAIN_SPLIT).exists(), f"Missing {MUSIC_TRAIN_SPLIT}"
assert SPEECH_VAL_SPLIT is not None or MUSIC_VAL_SPLIT is not None, "Need at least one validation/eval split for checkpoint selection."


def resolve_case_path(split_name: str, index: int):
    split_path = SPLIT_DIR / split_name
    assert split_path.exists(), f"Missing split file: {split_path}"
    items = [ln.strip().strip('"').strip("'") for ln in split_path.read_text().splitlines() if ln.strip()]
    assert 0 <= index < len(items), f"Index {index} out of range for {split_name}; len={len(items)}"
    return Path(items[index])


def read_split_paths(split_name: str):
    split_path = SPLIT_DIR / split_name
    assert split_path.exists(), f"Missing split file: {split_path}"
    return [Path(ln.strip().strip('"').strip("'")) for ln in split_path.read_text().splitlines() if ln.strip()]


def load_wav_segment(path: Path, seg_s: float, start_s: float = 0.0):
    wav, sr = sf.read(str(path), always_2d=False)
    wav = np.asarray(wav)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = torch.tensor(wav, dtype=torch.float32)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    start = int(round(start_s * SR))
    need = int(round(seg_s * SR))
    if wav.numel() < start + need:
        wav = F.pad(wav, (0, max(0, start + need - wav.numel())))
    wav = wav[start:start + need]
    peak = wav.abs().max().clamp_min(1e-8)
    if peak > 1.0:
        wav = wav / peak
    return wav.to(device)


def random_start_for_file(path: Path, seg_s: float):
    try:
        info = sf.info(str(path))
        dur = float(info.frames) / float(info.samplerate)
    except Exception:
        dur = seg_s
    max_start = max(0.0, dur - seg_s - 0.05)
    return random.uniform(0.0, max_start) if max_start > 0 else 0.0


def mask_1d(mask_mel: torch.Tensor):
    # accepts [B,1,1,T] or [B,1,T]
    if mask_mel.ndim == 4:
        return mask_mel[:, :, 0, :]
    return mask_mel


def dilate_mask_1d(mask_bt: torch.Tensor, radius: int):
    if radius <= 0:
        return mask_bt
    return F.max_pool1d(mask_bt, kernel_size=2 * radius + 1, stride=1, padding=radius)


def expand_mask_to_mel(mask_bt: torch.Tensor, n_mels: int):
    return mask_bt.expand(-1, n_mels, -1)


def match_audio_len(a, b):
    a = torch.as_tensor(a, device=device, dtype=torch.float32)
    b = torch.as_tensor(b, device=device, dtype=torch.float32)
    n = min(a.shape[-1], b.shape[-1])
    return a[..., :n], b[..., :n]


def to_np1(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().float().numpy()
    x = np.asarray(x, dtype=np.float32)
    return np.squeeze(x)


def safe_audio_np(x):
    arr = to_np1(x)
    arr = np.nan_to_num(arr)
    mx = float(np.max(np.abs(arr))) if arr.size else 0.0
    if mx > 1.0:
        arr = arr / mx
    return arr


def save_wav(name: str, wav, case_label: str):
    if not SAVE_AUDIO_FILES:
        return None
    safe_case = ''.join(c if c.isalnum() or c in '-_.' else '_' for c in case_label)[:120]
    safe_name = ''.join(c if c.isalnum() or c in '-_.' else '_' for c in name)[:120]
    d = OUT_DIR / safe_case
    d.mkdir(parents=True, exist_ok=True)
    p = d / f"{safe_name}.wav"
    sf.write(str(p), safe_audio_np(wav), SR)
    return p


def load_old_mel_model(spec, trainable=False):
    ck = torch_load_any(spec["ckpt_path"], map_location="cpu")
    rc = ck.get("run_config", ck.get("config", {}))
    frontend = ck.get("bigvgan_frontend_config", {})
    n_mels = int(frontend.get("num_mels", rc.get("n_mels", N_MELS)))
    base = int(rc.get("G_base", rc.get("g_base", G_BASE)))
    groups = int(rc.get("G_groups", rc.get("g_groups", G_GROUPS)))
    use_mask = bool(rc.get("G_use_mask", rc.get("g_use_mask", True)))
    G = PlainMelRefinerUNet2D(n_mels=n_mels, base=base, use_mask=use_mask, groups=groups).to(device)
    state = ck.get("G", ck.get("model", ck.get("generator", ck.get("state_dict", None))))
    assert state is not None, f"Could not find model weights in {spec['ckpt_path']}"
    G.load_state_dict(state, strict=True)
    G.train(bool(trainable))
    for p in G.parameters():
        p.requires_grad_(bool(trainable))
    print("Loaded old mel model:", spec["name"])
    print("step:", ck.get("step", "unknown"))
    print("base/groups/use_mask:", base, groups, use_mask)
    return G, ck

G_OLD, CK_OLD = load_old_mel_model(selected_spec, trainable=False)


In [ ]:

# =========================
# Linear-only item building and old model context
# =========================
@torch.no_grad()
def old_model_output_from_linear(mel_linear, mask_mel):
    m1 = mask_1d(mask_mel)
    delta = G_OLD(mel_linear, m1)
    mel_old = mel_linear + expand_mask_to_mel(m1, mel_linear.shape[1]) * delta
    return mel_old.detach()


def make_linear_item(path, split_name, source_type, start_s, seg_s, offset, include_tensors=True):
    wav = load_wav_segment(path, seg_s=seg_s, start_s=start_s)
    mel_real = wav_to_bigvgan_mel(wav.unsqueeze(0))
    mel_linear, mask_mel = linear_time_inpaint_mel(mel_real, k=K, offset=offset)
    mel_old = old_model_output_from_linear(mel_linear, mask_mel)
    item = {
        "path": str(path),
        "split": split_name,
        "source_type": source_type,
        "start_s": float(start_s),
        "seg_s": float(seg_s),
        "offset": int(offset),
    }
    if include_tensors:
        item.update({
            "mel_real": mel_real.detach().cpu(),
            "mel_linear": mel_linear.detach().cpu(),
            "mel_old": mel_old.detach().cpu(),
            "mask_mel": mask_mel.detach().cpu(),
        })
    return item


def item_tensors_to_device(item):
    return {
        "mel_real": item["mel_real"].to(device=device, dtype=torch.float32),
        "mel_linear": item["mel_linear"].to(device=device, dtype=torch.float32),
        "mel_old": item["mel_old"].to(device=device, dtype=torch.float32),
        "mask_mel": item["mask_mel"].to(device=device, dtype=torch.float32),
    }


def rebuild_item_from_meta(meta, include_tensors=True):
    return make_linear_item(
        Path(meta["path"]),
        split_name=meta["split"],
        source_type=meta["source_type"],
        start_s=float(meta["start_s"]),
        seg_s=float(meta.get("seg_s", SEG_S)),
        offset=int(meta["offset"]),
        include_tensors=include_tensors,
    )


In [ ]:

# =========================
# BigVGAN-residual mining diagnostic
# =========================
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

speech_train_paths = read_split_paths(SPEECH_TRAIN_SPLIT)
music_train_paths = read_split_paths(MUSIC_TRAIN_SPLIT)
speech_val_paths = read_split_paths(SPEECH_VAL_SPLIT) if SPEECH_VAL_SPLIT is not None else []
music_val_paths = read_split_paths(MUSIC_VAL_SPLIT) if MUSIC_VAL_SPLIT is not None else []
print("train speech/music:", len(speech_train_paths), len(music_train_paths))
print("val speech/music:", len(speech_val_paths), len(music_val_paths))


def choose_random_path(speech_paths, music_paths, speech_prob=SPEECH_PROB_MINE):
    use_speech = (random.random() < speech_prob) or not len(music_paths)
    if use_speech and len(speech_paths):
        return random.choice(speech_paths), "speech"
    return random.choice(music_paths), "music"


def stft_band_frame_rms(wav, low_hz=RESIDUAL_BAND_LOW_HZ, high_hz=RESIDUAL_BAND_HIGH_HZ, n_fft=1024, hop=256):
    if wav.ndim == 1:
        wav = wav.unsqueeze(0)
    win = torch.hann_window(n_fft, device=wav.device)
    X = torch.stft(wav, n_fft=n_fft, hop_length=hop, win_length=n_fft, window=win, return_complex=True, center=True)
    mag2 = X.abs().pow(2)[0]
    freqs = torch.linspace(0, SR/2, mag2.shape[0], device=wav.device)
    band = (freqs >= low_hz) & (freqs <= high_hz)
    if int(band.sum()) == 0:
        band[:] = True
    frame = torch.sqrt(mag2[band].mean(dim=0).clamp_min(1e-12))
    return frame


@torch.no_grad()
def bigvgan_residual_score_for_item(item):
    x = item_tensors_to_device(item)
    wav_ref = mel_to_wave_bigvgan(x["mel_real"])[0].detach().clamp(-1, 1)
    scores = {}
    for name, mel in [("linear", x["mel_linear"]), ("old", x["mel_old"] )]:
        wav_bad = mel_to_wave_bigvgan(mel)[0].detach().clamp(-1, 1)
        wb, wr = match_audio_len(wav_bad, wav_ref)
        res = wb - wr
        frame = stft_band_frame_rms(res)
        rms = torch.sqrt(torch.mean(res ** 2)).item()
        peak = res.abs().max().item()
        q95 = torch.quantile(frame, 0.95).item()
        q99 = torch.quantile(frame, 0.99).item()
        mx = frame.max().item()
        # q99/max catch short sharp buzz-like events. rms keeps very tiny numerical spikes from dominating.
        score = q99 + 0.35 * mx + 0.50 * rms
        scores.update({
            f"{name}_res_rms": float(rms),
            f"{name}_res_peak": float(peak),
            f"{name}_band_q95": float(q95),
            f"{name}_band_q99": float(q99),
            f"{name}_band_max": float(mx),
            f"{name}_score": float(score),
        })
    selected_scores = []
    if RESIDUAL_SCORE_USE_LINEAR:
        selected_scores.append(scores["linear_score"])
    if RESIDUAL_SCORE_USE_OLD:
        selected_scores.append(scores["old_score"])
    scores["score"] = float(max(selected_scores)) if selected_scores else float(scores["old_score"])
    return scores


def mine_split_bigvgan_residual(split_name, speech_paths, music_paths, n_candidates, speech_prob, tag):
    metas = []
    for i in tqdm(range(n_candidates), desc=f"BigVGAN residual mining {tag}"):
        try:
            path, source_type = choose_random_path(speech_paths, music_paths, speech_prob=speech_prob)
            start_s = random_start_for_file(path, SEG_S)
            offset = random.choice(TRAIN_OFFSETS)
            item = make_linear_item(path, split_name=split_name, source_type=source_type, start_s=start_s, seg_s=SEG_S, offset=offset, include_tensors=True)
            scores = bigvgan_residual_score_for_item(item)
            meta = {k: item[k] for k in ["path", "split", "source_type", "start_s", "seg_s", "offset"]}
            meta.update(scores)
            metas.append(meta)
        except RuntimeError as e:
            print("RuntimeError during mining; skipping candidate", i, repr(e))
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print("Error during mining; skipping candidate", i, repr(e))
        if torch.cuda.is_available() and (i + 1) % 25 == 0:
            torch.cuda.empty_cache()
    metas = sorted(metas, key=lambda d: d.get("score", -1), reverse=True)
    return metas


def run_or_load_residual_mining():
    if RESIDUAL_CACHE_PATH.exists() and not REBUILD_RESIDUAL_CACHE:
        print("Loading residual mining cache:", RESIDUAL_CACHE_PATH)
        return torch_load_any(RESIDUAL_CACHE_PATH, map_location="cpu")

    train_metas = mine_split_bigvgan_residual(
        split_name="train_mixed",
        speech_paths=speech_train_paths,
        music_paths=music_train_paths,
        n_candidates=DIAG_MINE_TRAIN_CANDIDATES,
        speech_prob=SPEECH_PROB_MINE,
        tag="train",
    )
    val_metas = mine_split_bigvgan_residual(
        split_name="val_mixed",
        speech_paths=speech_val_paths,
        music_paths=music_val_paths,
        n_candidates=DIAG_MINE_VAL_CANDIDATES,
        speech_prob=SPEECH_PROB_MINE,
        tag="val",
    )
    cache = {
        "config": {
            "SEG_S": SEG_S,
            "DIAG_MINE_TRAIN_CANDIDATES": DIAG_MINE_TRAIN_CANDIDATES,
            "DIAG_MINE_VAL_CANDIDATES": DIAG_MINE_VAL_CANDIDATES,
            "RESIDUAL_BAND_LOW_HZ": RESIDUAL_BAND_LOW_HZ,
            "RESIDUAL_BAND_HIGH_HZ": RESIDUAL_BAND_HIGH_HZ,
            "old_checkpoint": str(selected_spec["ckpt_path"]),
        },
        "train_metas": train_metas,
        "val_metas": val_metas,
    }
    torch.save(cache, str(RESIDUAL_CACHE_PATH))
    print("Saved residual mining cache:", RESIDUAL_CACHE_PATH)
    return cache

residual_cache = run_or_load_residual_mining()
train_df = pd.DataFrame(residual_cache["train_metas"])
val_df = pd.DataFrame(residual_cache["val_metas"])
print("train mined:", len(train_df), "val mined:", len(val_df))

display(Markdown("## Top train examples mined by actual BigVGAN residual"))
display(train_df.head(20))
display(Markdown("## Top validation examples mined by actual BigVGAN residual"))
display(val_df.head(20))

for title, df in [("train", train_df), ("val", val_df)]:
    if len(df):
        plt.figure(figsize=(9,3))
        plt.hist(df["score"].values, bins=40)
        plt.title(f"{title} BigVGAN residual score distribution")
        plt.xlabel("score")
        plt.ylabel("count")
        plt.grid(True, alpha=0.25)
        plt.show()


In [ ]:

# =========================
# Listen to top residual-mined examples before training anything
# =========================
def residual_metrics_audio(wav_bad, wav_ref):
    wb, wr = match_audio_len(wav_bad, wav_ref)
    res = wb - wr
    frame = stft_band_frame_rms(res)
    return {
        "res_rms": float(torch.sqrt(torch.mean(res**2)).detach().cpu()),
        "res_peak": float(res.abs().max().detach().cpu()),
        "band_q95": float(torch.quantile(frame, 0.95).detach().cpu()),
        "band_q99": float(torch.quantile(frame, 0.99).detach().cpu()),
        "band_max": float(frame.max().detach().cpu()),
    }


def masked_mel_metrics_simple(mel_hat, mel_real, mask_mel):
    m = expand_mask_to_mel(mask_1d(mask_mel), mel_real.shape[1])
    denom = m.sum().clamp_min(1.0)
    e = (mel_hat - mel_real).abs()
    hf_start = int(0.45 * mel_real.shape[1])
    vhf_start = int(0.70 * mel_real.shape[1])
    return {
        "mask_l1": float((e*m).sum().detach().cpu() / denom.detach().cpu()),
        "hf_l1": float((e[:, hf_start:, :]*m[:, hf_start:, :]).sum().detach().cpu() / m[:, hf_start:, :].sum().clamp_min(1.0).detach().cpu()),
        "vhf_l1": float((e[:, vhf_start:, :]*m[:, vhf_start:, :]).sum().detach().cpu() / m[:, vhf_start:, :].sum().clamp_min(1.0).detach().cpu()),
    }


def save_audio_eval(case_label, name, wav):
    return save_wav(name, wav, case_label)


@torch.no_grad()
def listen_to_item(item_or_meta, title="example"):
    if "mel_real" not in item_or_meta:
        item = rebuild_item_from_meta(item_or_meta, include_tensors=True)
    else:
        item = item_or_meta
    x = item_tensors_to_device(item)
    label = f"{title} · {item['source_type']} · start={item['start_s']:.3f}s · offset={item['offset']}"
    display(Markdown(f"# {label}"))
    print("path:", item["path"])
    print("score meta:", {k: item_or_meta.get(k) for k in ["score", "linear_score", "old_score", "linear_band_q99", "old_band_q99"] if isinstance(item_or_meta, dict) and k in item_or_meta})

    mel_dict = {
        "00 original mel / oracle": x["mel_real"],
        "01 linear interp": x["mel_linear"],
        "02 old 2D U-Net": x["mel_old"],
    }
    wav_ref = mel_to_wave_bigvgan(x["mel_real"])[0].detach().clamp(-1, 1)
    rows = []
    wavs = {}
    for name, mel in mel_dict.items():
        wav = mel_to_wave_bigvgan(mel)[0].detach().clamp(-1, 1)
        wavs[name] = wav
        row = {"output": name}
        row.update(masked_mel_metrics_simple(mel, x["mel_real"], x["mask_mel"]))
        row.update(residual_metrics_audio(wav, wav_ref))
        rows.append(row)
        save_audio_eval(label, name, wav)

    display(pd.DataFrame(rows))
    for name, wav in wavs.items():
        display(Markdown(f"**{name} → BigVGAN**"))
        display(Audio(safe_audio_np(wav), rate=SR))

    # Listen to boosted residuals. This is not for quality, just for identifying the artifact.
    for name in ["01 linear interp", "02 old 2D U-Net"]:
        wb, wr = match_audio_len(wavs[name], wav_ref)
        res = (wb - wr).detach().clamp(-1, 1)
        display(Markdown(f"**Residual: {name} − original mel→BigVGAN, boosted ×6**"))
        display(Audio(safe_audio_np((6.0 * res).clamp(-1, 1)), rate=SR))
    return item

# Listen to a few top TRAIN examples. These are allowed for mining/training.
TOP_TRAIN_ITEMS = []
for i, meta in enumerate(residual_cache["train_metas"][:TOPK_TRAIN_LISTEN]):
    TOP_TRAIN_ITEMS.append(listen_to_item(meta, title=f"TRAIN residual-mined #{i}"))

# Listen to a few top VAL examples. These are for checkpoint choice, not training.
TOP_VAL_ITEMS = []
for i, meta in enumerate(residual_cache["val_metas"][:TOPK_VAL_LISTEN]):
    TOP_VAL_ITEMS.append(listen_to_item(meta, title=f"VAL residual-mined #{i}"))


In [ ]:

# =========================
# Adapter model
# =========================
class AdapterConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8, dropout=0.0):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        ]
        if dropout and dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class AdapterUp(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            AdapterConvGNAct(in_ch + skip_ch, out_ch, groups=groups, dropout=dropout),
            AdapterConvGNAct(out_ch, out_ch, groups=groups, dropout=dropout),
        )
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class MissingFrameAdapterUNet(nn.Module):
    """Small residual adapter that predicts a correction on top of linear interpolated mel."""
    def __init__(self, in_ch=6, base=32, groups=8, dropout=0.05):
        super().__init__()
        self.e0 = nn.Sequential(
            AdapterConvGNAct(in_ch, base, groups=groups, dropout=dropout),
            AdapterConvGNAct(base, base, groups=groups, dropout=dropout),
        )
        self.e1 = nn.Sequential(
            AdapterConvGNAct(base, base*2, s=(2,2), groups=groups, dropout=dropout),
            AdapterConvGNAct(base*2, base*2, groups=groups, dropout=dropout),
        )
        self.e2 = nn.Sequential(
            AdapterConvGNAct(base*2, base*4, s=(2,2), groups=groups, dropout=dropout),
            AdapterConvGNAct(base*4, base*4, groups=groups, dropout=dropout),
        )
        self.e3 = nn.Sequential(
            AdapterConvGNAct(base*4, base*6, s=(2,2), groups=groups, dropout=dropout),
            AdapterConvGNAct(base*6, base*6, groups=groups, dropout=dropout),
        )
        # Dilated temporal/frequency context at bottleneck.
        self.bot = nn.Sequential(
            AdapterConvGNAct(base*6, base*6, k=3, p=1, groups=groups, dropout=dropout),
            nn.Conv2d(base*6, base*6, kernel_size=3, padding=2, dilation=2),
            nn.GroupNorm(_valid_groups(base*6, groups), base*6),
            nn.SiLU(inplace=True),
            nn.Conv2d(base*6, base*6, kernel_size=3, padding=4, dilation=4),
            nn.GroupNorm(_valid_groups(base*6, groups), base*6),
            nn.SiLU(inplace=True),
        )
        self.u2 = AdapterUp(base*6, base*4, base*4, groups=groups, dropout=dropout)
        self.u1 = AdapterUp(base*4, base*2, base*2, groups=groups, dropout=dropout)
        self.u0 = AdapterUp(base*2, base, base, groups=groups, dropout=dropout)
        self.out = nn.Conv2d(base, 1, kernel_size=3, padding=1)
    def forward(self, x):
        s0 = self.e0(x)
        s1 = self.e1(s0)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        b = self.bot(s3)
        u2 = self.u2(b, s2)
        u1 = self.u1(u2, s1)
        u0 = self.u0(u1, s0)
        return self.out(u0).squeeze(1)

ADAPTER = MissingFrameAdapterUNet(in_ch=ADAPTER_IN_CH, base=ADAPTER_BASE, groups=ADAPTER_GROUPS, dropout=ADAPTER_DROPOUT).to(device)
print(ADAPTER)
print("trainable params:", sum(p.numel() for p in ADAPTER.parameters() if p.requires_grad))


In [ ]:

# =========================
# Adapter input/loss/training utilities: regular linear interpolation only
# =========================
def make_adapter_input_linear(mel_linear, mel_old, mask_mel):
    m = expand_mask_to_mel(mask_1d(mask_mel), mel_linear.shape[1])
    old_delta = mel_old - mel_linear
    # Shape [B, C, F, T]
    return torch.stack([mel_linear, mel_old, old_delta, m], dim=1)


def adapter_forward_linear(adapter, batch):
    x = make_adapter_input_linear(batch["mel_linear"], batch["mel_old"], batch["mask_mel"])
    delta = adapter(x)
    m = expand_mask_to_mel(mask_1d(batch["mask_mel"]), batch["mel_linear"].shape[1])
    mel_hat = batch["mel_linear"] + m * delta
    return mel_hat, delta


def batch_from_items(items):
    xs = [item_tensors_to_device(it) for it in items]
    minT = min(x["mel_real"].shape[-1] for x in xs)
    batch = {}
    for key in ["mel_real", "mel_linear", "mel_old", "mask_mel"]:
        batch[key] = torch.cat([x[key][..., :minT] for x in xs], dim=0)
    return batch


def make_local_frame_weight_from_old_error(mel_real, mel_old, mask_mel):
    m = expand_mask_to_mel(mask_1d(mask_mel), mel_real.shape[1])
    hf_start = int(OVERFIT_HF_START_FRAC * mel_real.shape[1])
    e = (mel_old - mel_real).abs() * m
    frame = e[:, hf_start:, :].mean(dim=1, keepdim=True)  # [B,1,T]
    frame = frame / (frame.mean(dim=-1, keepdim=True).clamp_min(1e-6))
    frame = frame.clamp(0.0, 8.0)
    frame = 1.0 + LOCAL_FRAME_GAIN * (frame / frame.max(dim=-1, keepdim=True).values.clamp_min(1e-6)).pow(LOCAL_FRAME_POWER)
    return frame


def local_masked_mean(x, mask_ft, weights=None, frame_weight=None, eps=1e-8):
    z = x.abs()
    denom = mask_ft
    if weights is not None:
        z = z * weights
        denom = denom * weights
    if frame_weight is not None:
        z = z * frame_weight
        denom = denom * frame_weight
    return (z * mask_ft).sum() / (denom.sum() + eps)


def compute_linear_adapter_losses(adapter, batch):
    mel_hat, delta = adapter_forward_linear(adapter, batch)
    mel_real = batch["mel_real"]
    mel_linear = batch["mel_linear"]
    mask_mel = batch["mask_mel"]
    mask_ft = expand_mask_to_mel(mask_1d(mask_mel), mel_real.shape[1])
    frame_w = make_local_frame_weight_from_old_error(mel_real, batch["mel_old"], mask_mel)

    hf_w = build_freq_weights(mel_real.shape[1], OVERFIT_HF_START_FRAC, OVERFIT_HF_END_GAIN, OVERFIT_HF_POWER, device=mel_real.device)
    vhf_w = build_freq_weights(mel_real.shape[1], OVERFIT_VHF_START_FRAC, OVERFIT_HF_END_GAIN, OVERFIT_HF_POWER, device=mel_real.device)

    err = mel_hat - mel_real
    loss_l1 = local_masked_mean(err, mask_ft, frame_weight=frame_w)
    loss_l2 = ((err ** 2) * mask_ft * frame_w).sum() / ((mask_ft * frame_w).sum() + 1e-8)
    loss_hf = local_masked_mean(err, mask_ft, weights=hf_w, frame_weight=frame_w)
    loss_vhf = local_masked_mean(err, mask_ft, weights=vhf_w, frame_weight=frame_w)

    # Time derivative losses, focused near missing frames.
    d_hat = mel_hat[..., 1:] - mel_hat[..., :-1]
    d_real = mel_real[..., 1:] - mel_real[..., :-1]
    d_mask = torch.maximum(mask_ft[..., 1:], mask_ft[..., :-1])
    d_fw = frame_w[..., 1:]
    loss_td = local_masked_mean(d_hat - d_real, d_mask, weights=hf_w, frame_weight=d_fw)

    dd_hat = mel_hat[..., 2:] - 2 * mel_hat[..., 1:-1] + mel_hat[..., :-2]
    dd_real = mel_real[..., 2:] - 2 * mel_real[..., 1:-1] + mel_real[..., :-2]
    dd_mask = torch.maximum(torch.maximum(mask_ft[..., 2:], mask_ft[..., 1:-1]), mask_ft[..., :-2])
    dd_fw = frame_w[..., 2:]
    loss_dd = local_masked_mean(dd_hat - dd_real, dd_mask, weights=hf_w, frame_weight=dd_fw)

    # Delta target: learn the correction that turns linear interpolation into true mel.
    target_delta = mel_real - mel_linear
    loss_delta = local_masked_mean(delta - target_delta, mask_ft, weights=hf_w, frame_weight=frame_w)

    # Peak/top-frame loss: force short sharp local mistakes to matter.
    with torch.no_grad():
        local_e = ((batch["mel_old"] - mel_real).abs() * mask_ft)[:, int(OVERFIT_HF_START_FRAC*mel_real.shape[1]):, :].mean(dim=1)
        k = max(1, int(PEAK_TOP_FRAC * local_e.shape[-1]))
        top_idx = torch.topk(local_e, k=k, dim=-1).indices
        peak_mask = torch.zeros_like(local_e)
        peak_mask.scatter_(1, top_idx, 1.0)
        peak_mask = peak_mask.unsqueeze(1)
        peak_mask = peak_mask.expand(-1, mel_real.shape[1], -1)
    loss_peak = local_masked_mean(err, peak_mask, weights=hf_w)

    # Keep adapter from making wild corrections everywhere.
    loss_reg = local_masked_mean(delta, mask_ft)

    loss = (
        LOSS_W_L1 * loss_l1 +
        LOSS_W_L2 * loss_l2 +
        LOSS_W_HF * loss_hf +
        LOSS_W_VHF * loss_vhf +
        LOSS_W_TDIFF * loss_td +
        LOSS_W_DDIFF * loss_dd +
        LOSS_W_DELTA * loss_delta +
        LOSS_W_PEAK * loss_peak +
        LOSS_W_RESID_REG * loss_reg
    )
    metrics = {
        "loss": float(loss.detach().cpu()),
        "l1": float(loss_l1.detach().cpu()),
        "l2": float(loss_l2.detach().cpu()),
        "hf": float(loss_hf.detach().cpu()),
        "vhf": float(loss_vhf.detach().cpu()),
        "tdiff": float(loss_td.detach().cpu()),
        "ddiff": float(loss_dd.detach().cpu()),
        "delta": float(loss_delta.detach().cpu()),
        "peak": float(loss_peak.detach().cpu()),
        "reg": float(loss_reg.detach().cpu()),
    }
    return loss, metrics, mel_hat


def evaluate_adapter_cached(adapter, items, n=None, title="cached eval"):
    if n is not None:
        items = items[:n]
    rows = []
    adapter.eval()
    with torch.no_grad():
        for i, item in enumerate(items):
            batch = batch_from_items([item])
            loss, metrics, mel_hat = compute_linear_adapter_losses(adapter, batch)
            row = {"i": i, **metrics}
            row.update({"path": item["path"], "start_s": item["start_s"], "source_type": item["source_type"]})
            rows.append(row)
    display(Markdown(f"## {title}"))
    display(pd.DataFrame(rows))
    return pd.DataFrame(rows)


In [ ]:

# =========================
# Optional: single-example overfit on TOP TRAIN example, not test
# =========================
if RUN_SINGLE_EXAMPLE_OVERFIT:
    assert len(TOP_TRAIN_ITEMS) > 0, "Need TOP_TRAIN_ITEMS from the listening cell."
    train_item = TOP_TRAIN_ITEMS[0]
    ADAPTER_SINGLE = MissingFrameAdapterUNet(in_ch=ADAPTER_IN_CH, base=ADAPTER_BASE, groups=ADAPTER_GROUPS, dropout=0.0).to(device)
    opt = torch.optim.AdamW(ADAPTER_SINGLE.parameters(), lr=SINGLE_OVERFIT_LR, weight_decay=1e-6)
    print("Single-example overfit on:", train_item["path"], "start", train_item["start_s"])
    for step in range(1, SINGLE_OVERFIT_STEPS + 1):
        batch = batch_from_items([train_item])
        opt.zero_grad(set_to_none=True)
        loss, metrics, mel_hat = compute_linear_adapter_losses(ADAPTER_SINGLE, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ADAPTER_SINGLE.parameters(), 1.0)
        opt.step()
        if step == 1 or step % 250 == 0 or step == SINGLE_OVERFIT_STEPS:
            print("step", step, {k: round(v, 5) for k, v in metrics.items()})
    # Listen after single-example overfit.
    batch = batch_from_items([train_item])
    with torch.no_grad():
        _, _, mel_adapt = compute_linear_adapter_losses(ADAPTER_SINGLE, batch)
        wav_ref = mel_to_wave_bigvgan(batch["mel_real"])[0].detach().clamp(-1, 1)
        outputs = {
            "00 original mel / oracle": batch["mel_real"],
            "01 linear interp": batch["mel_linear"],
            "02 old 2D U-Net": batch["mel_old"],
            "03 single-overfit adapter": mel_adapt.detach(),
        }
        rows=[]
        for name, mel in outputs.items():
            wav = mel_to_wave_bigvgan(mel)[0].detach().clamp(-1,1)
            row={"output": name}
            row.update(masked_mel_metrics_simple(mel, batch["mel_real"], batch["mask_mel"]))
            row.update(residual_metrics_audio(wav, wav_ref))
            rows.append(row)
            display(Markdown(f"**{name} → BigVGAN**"))
            display(Audio(safe_audio_np(wav), rate=SR))
        display(pd.DataFrame(rows))
else:
    print("RUN_SINGLE_EXAMPLE_OVERFIT=False. Skipping single-example overfit in the long run.")


## Training run notes 
This run does not use the test clips for training. It mines only train/validation examples by actual BigVGAN residual, then trains a linear-interpolation adapter on true missing mel frames. The final cell evaluates the familiar speech/music clips only after training.

The intended outcome is that the validation residual-mined examples sound cleaner and that the held-out clips improve relative to the earlier 2D U-Net. This is the most targeted run in the set, while remaining an experimental configuration rather than a guaranteed fix.

In [ ]:

# =========================
# long run adapter training on BigVGAN-residual-mined train examples
# =========================
def save_pilot_checkpoint(adapter, step, is_latest=False):
    payload = {
        "step": int(step),
        "adapter": adapter.state_dict(),
        "run_config": {
            "RUN_NAME": RUN_NAME,
            "ADAPTER_IN_CH": ADAPTER_IN_CH,
            "ADAPTER_BASE": ADAPTER_BASE,
            "ADAPTER_GROUPS": ADAPTER_GROUPS,
            "ADAPTER_DROPOUT": ADAPTER_DROPOUT,
            "K": K,
            "SEG_S": SEG_S,
            "base_interp": "linear_log_mel",
            "old_checkpoint": str(selected_spec["ckpt_path"]),
            "mining": "BigVGAN audio residual",
        },
    }
    p = RUN_DIR / ("linear_adapter_residual_mined_latest.pt" if is_latest else f"linear_adapter_residual_mined_step{step:06d}.pt")
    torch.save(payload, str(p))
    return p


def sample_pilot_batch(train_items):
    items = random.sample(train_items, k=min(PILOT_BATCH_SIZE, len(train_items)))
    return batch_from_items(items)

if RUN_SHORT_PILOT_TRAINING:
    # Rebuild top residual-mined items as tensor cache.
    N_PILOT_TRAIN = min(N_RESIDUAL_TRAIN_BANK, len(residual_cache["train_metas"]))
    N_PILOT_VAL = min(N_RESIDUAL_VAL_BANK, len(residual_cache["val_metas"]))
    PILOT_TRAIN_ITEMS = [rebuild_item_from_meta(m, include_tensors=True) for m in residual_cache["train_metas"][:N_PILOT_TRAIN]]
    PILOT_VAL_ITEMS = [rebuild_item_from_meta(m, include_tensors=True) for m in residual_cache["val_metas"][:N_PILOT_VAL]]
    print("long run train/val residual-mined items:", len(PILOT_TRAIN_ITEMS), len(PILOT_VAL_ITEMS))

    ADAPTER_PILOT = MissingFrameAdapterUNet(in_ch=ADAPTER_IN_CH, base=ADAPTER_BASE, groups=ADAPTER_GROUPS, dropout=ADAPTER_DROPOUT).to(device)
    opt = torch.optim.AdamW(ADAPTER_PILOT.parameters(), lr=PILOT_LR, weight_decay=1e-5)
    hist=[]
    for step in range(1, PILOT_STEPS + 1):
        ADAPTER_PILOT.train()
        batch = sample_pilot_batch(PILOT_TRAIN_ITEMS)
        opt.zero_grad(set_to_none=True)
        loss, metrics, _ = compute_linear_adapter_losses(ADAPTER_PILOT, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ADAPTER_PILOT.parameters(), 1.0)
        opt.step()
        row={"step": step, **metrics}
        if step == 1 or step % PILOT_VAL_EVERY == 0:
            ADAPTER_PILOT.eval()
            vals=[]
            with torch.no_grad():
                for _ in range(4):
                    vb = batch_from_items(random.sample(PILOT_VAL_ITEMS, k=min(PILOT_BATCH_SIZE, len(PILOT_VAL_ITEMS))))
                    _, vm, _ = compute_linear_adapter_losses(ADAPTER_PILOT, vb)
                    vals.append(vm)
            vmean={"val_"+k: float(np.mean([v[k] for v in vals])) for k in vals[0]}
            row.update(vmean)
            print("step", step, {k: round(v,5) for k,v in row.items() if k != "step"})
        hist.append(row)
        if step % PILOT_SAVE_EVERY == 0 or step == PILOT_STEPS:
            save_pilot_checkpoint(ADAPTER_PILOT, step, is_latest=False)
            save_pilot_checkpoint(ADAPTER_PILOT, step, is_latest=True)
            pd.DataFrame(hist).to_csv(LOG_PATH, index=False)
    pd.DataFrame(hist).to_csv(LOG_PATH, index=False)
    print("Pilot done. Latest:", RUN_DIR / "linear_adapter_residual_mined_latest.pt")
else:
    print("RUN_SHORT_PILOT_TRAINING=False. Set it True to launch the long adapter training.")


In [ ]:

# =========================
# Optional evaluation of residual-mined adapter on familiar diagnostic clips
# Does NOT train or mine on these clips.
# =========================
def load_pilot_adapter(ckpt_path=None):
    if ckpt_path is None:
        ckpt_path = RUN_DIR / "linear_adapter_residual_mined_latest.pt"
    assert Path(ckpt_path).exists(), f"Missing adapter checkpoint: {ckpt_path}"
    ck = torch_load_any(ckpt_path, map_location="cpu")
    cfg = ck.get("run_config", {})
    A = MissingFrameAdapterUNet(
        in_ch=int(cfg.get("ADAPTER_IN_CH", ADAPTER_IN_CH)),
        base=int(cfg.get("ADAPTER_BASE", ADAPTER_BASE)),
        groups=int(cfg.get("ADAPTER_GROUPS", ADAPTER_GROUPS)),
        dropout=float(cfg.get("ADAPTER_DROPOUT", ADAPTER_DROPOUT)),
    ).to(device)
    A.load_state_dict(ck["adapter"], strict=True)
    A.eval()
    return A, ck


def build_eval_case(case):
    path = resolve_case_path(case["split"], case["index"])
    item = make_linear_item(path, split_name=case["split"], source_type="test", start_s=float(case.get("start_s", 0.0)), seg_s=float(case["seg_s"]), offset=int(case.get("offset", EVAL_OFFSET)), include_tensors=True)
    item["label"] = case["label"]
    return item


def evaluate_adapter_on_case(adapter, case):
    item = build_eval_case(case)
    batch = batch_from_items([item])
    with torch.no_grad():
        mel_adapter, _ = adapter_forward_linear(adapter, batch)
        wav_ref = mel_to_wave_bigvgan(batch["mel_real"])[0].detach().clamp(-1, 1)
        outputs = {
            "00 original mel / oracle": batch["mel_real"],
            "01 linear interp": batch["mel_linear"],
            "02 old 2D U-Net": batch["mel_old"],
            "03 residual-mined adapter": mel_adapter.detach(),
        }
        rows=[]
        display(Markdown(f"# Familiar diagnostic only: {item['label']}"))
        print("path:", item["path"])
        for name, mel in outputs.items():
            wav = mel_to_wave_bigvgan(mel)[0].detach().clamp(-1,1)
            row={"output": name}
            row.update(masked_mel_metrics_simple(mel, batch["mel_real"], batch["mask_mel"]))
            row.update(residual_metrics_audio(wav, wav_ref))
            rows.append(row)
            display(Markdown(f"**{name} → BigVGAN**"))
            display(Audio(safe_audio_np(wav), rate=SR))
        display(pd.DataFrame(rows))

if (RUN_SHORT_PILOT_TRAINING or (RUN_DIR / "linear_adapter_residual_mined_latest.pt").exists()):
    try:
        A_PILOT, CK_PILOT = load_pilot_adapter()
        print("Loaded residual-mined adapter step:", CK_PILOT.get("step"))
        for case in FINAL_TEST_CASES:
            evaluate_adapter_on_case(A_PILOT, case)
    except AssertionError as e:
        print(e)
else:
    print("No residual-mined adapter checkpoint yet. This cell is for after optional pilot training.")
